# 🐟 fishaudio/s2-pro — Long-Form Hinglish Audiobook Pipeline
## L4 GPU · Smart Chunking · Voice Cloning

**Model**: `fishaudio/s2-pro` (4B Dual-AR)
**Runtime**: Google Colab Pro+ · **NVIDIA L4 GPU** (24 GB VRAM)
**Features**:
- 🎯 GPU-accelerated inference (CUDA)
- 📖 Smart sentence-boundary text chunking for long stories
- 🎙️ Reference audio voice cloning across all chunks
- 🎭 Emotion tags: `[excited]` `[whisper]` `[pause]` `[laugh]` etc.
- 🌐 Hinglish (Devanagari + Latin script) fully supported
- 📊 Real-time Iron Man themed progress dashboard

**Workflow**: Cell 1→8 in order. No restarts needed.

| Cell | Purpose |
|------|--------|
| 1 | Environment & GPU audit |
| 2 | Configuration |
| 3 | Upload text file (.txt) |
| 4 | Upload reference audio (optional) |
| 5 | Install fish-speech & deps |
| 6 | Download model weights |
| 7 | **S2-PRO Inference** (chunked GPU) |
| 8 | Play & download output |


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT & HARDWARE AUDIT
# ════════════════════════════════════════════════════════════
import os, sys, subprocess, platform, time

# Set CPU threads for non-GPU workloads
os.environ['OMP_NUM_THREADS']  = '8'
os.environ['MKL_NUM_THREADS']  = '8'

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 1 — ENVIRONMENT & HARDWARE AUDIT")
print(f"{SEP}\n")

print("🐍 PYTHON")
print(f"   Version      : {sys.version.split()[0]}")
print(f"   Executable   : {sys.executable}")
print(f"   Platform     : {platform.platform()}")
print(f"   Architecture : {platform.machine()}")

print(f"\n💻 CPU")
try:
    with open('/proc/cpuinfo') as f:
        cpuinfo = f.read()
    model_lines  = [l for l in cpuinfo.splitlines() if 'model name' in l]
    cpu_model    = model_lines[0].split(':')[1].strip() if model_lines else 'Unknown'
    cpu_logical  = cpuinfo.count('processor\t:')
    print(f"   Model        : {cpu_model}")
    print(f"   Logical cores: {cpu_logical}")
except Exception as e:
    print(f"   /proc/cpuinfo: {e}")

print(f"\n🧠 HOST RAM")
try:
    with open('/proc/meminfo') as f:
        mi = {l.split(':')[0]: l.split(':')[1].strip() for l in f}
    total_gb = int(mi['MemTotal'].split()[0])     / 1e6
    avail_gb = int(mi['MemAvailable'].split()[0]) / 1e6
    print(f"   Total        : {total_gb:.1f} GB")
    print(f"   Available    : {avail_gb:.1f} GB")
    if avail_gb < 10:
        print(f"   ⚠️  Less than 10 GB available — model load may be tight.")
    else:
        print(f"   ✅ Sufficient for model loading.")
except Exception as e:
    print(f"   /proc/meminfo: {e}")
    total_gb = 0

print(f"\n💾 DISK SPACE")
try:
    st = os.statvfs('/')
    disk_total_gb = (st.f_blocks * st.f_frsize) / 1e9
    disk_free_gb  = (st.f_bavail * st.f_frsize) / 1e9
    print(f"   Total        : {disk_total_gb:.1f} GB")
    print(f"   Free         : {disk_free_gb:.1f} GB")
    print(f"   ✅ OK" if disk_free_gb >= 15 else f"   ⚠️  Less than 15 GB free.")
except Exception as e:
    print(f"   statvfs: {e}")

# ── GPU check ────────────────────────────────────────────────
print(f"\n⚡ ACCELERATOR")
_has_gpu = False
try:
    smi = subprocess.check_output(
        ['nvidia-smi','--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader,nounits'],
        stderr=subprocess.DEVNULL).decode().strip()
    parts = [p.strip() for p in smi.split(",")]
    gpu_name = parts[0] if parts else "Unknown"
    gpu_vram = parts[1] if len(parts) > 1 else "?"
    gpu_driver = parts[2] if len(parts) > 2 else "?"
    print(f"   GPU          : {gpu_name}")
    print(f"   VRAM         : {gpu_vram} MB")
    print(f"   Driver       : {gpu_driver}")
    vram_mb = int(gpu_vram)
    _has_gpu = True
    if vram_mb >= 22000:
        print(f"   ✅ Sufficient VRAM for s2-pro (~17 GB needed)")
    elif vram_mb >= 14000:
        print(f"   ⚠️  Tight VRAM — may work but watch for OOM")
    else:
        print(f"   ⚠️  Low VRAM — will fall back to CPU inference")
        _has_gpu = False
except Exception:
    print(f"   GPU          : none detected")

# ── TPU check ────────────────────────────────────────────────
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print(f"   torch_xla    : {torch_xla.__version__}")
    print(f"   TPU devices  : {xm.get_xla_supported_devices()}")
    print(f"   ✅ TPU detected — host RAM ({total_gb:.0f} GB) used for model weights")
    if not _has_gpu:
        print(f"   → Will use CPU inference with TPU's high-RAM host")
except ImportError:
    if not _has_gpu:
        print(f"   ℹ️  No GPU or TPU — CPU inference (slow)")

print(f"\n🔧 TOOLS")
for tool in ['git','ffmpeg','python3','pip3','curl']:
    try:
        path = subprocess.check_output(['which',tool],stderr=subprocess.DEVNULL).decode().strip()
        print(f"   ✅ {tool:12s} → {path}")
    except:
        print(f"   ❌ {tool:12s} → NOT FOUND")

print(f"\n🔥 PYTORCH")
try:
    import torch
    print(f"   Version      : {torch.__version__}")
    print(f"   CUDA avail   : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA device  : {torch.cuda.get_device_name(0)}")
        print(f"   CUDA version : {torch.version.cuda}")
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"   VRAM (torch) : {vram:.1f} GB")
        print(f"   ✅ GPU ready — Cell 2 will auto-detect DEVICE='cuda'")
    else:
        print(f"   ℹ️  CUDA not available — Cell 2 will auto-detect DEVICE='cpu'")
    torch.set_num_threads(8)
    print(f"   CPU threads  → 8")
except ImportError:
    print(f"   Not installed yet (done in Cell 5)")

print(f"\n{SEP}")
print("  ✅ CELL 1 COMPLETE — Proceed to Cell 2")
print(f"{SEP}")


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ════════════════════════════════════════════════════════════
# Edit variables below. All subsequent cells read from here.
# ════════════════════════════════════════════════════════════
import os

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 2 — CONFIGURATION")
print(f"{SEP}\n")

# ── EDIT THESE ────────────────────────────────────────────────
DO_INSTALL     = True           # False = skip install (if already done)
MODEL_DIR      = '/content/checkpoints/s2-pro'
OUTPUT_DIR     = '/content/inference_outputs'
HF_TOKEN       = ''             # Leave empty for public model access

# ── DEVICE — auto-detect (works on both v5e-1 and L4) ─────────
# 'auto' = try CUDA first, fall back to CPU
# 'cuda' = force GPU (L4)    'cpu' = force CPU (v5e-1 / T4)
DEVICE         = 'auto'

# ── VOICE CLONING ─────────────────────────────────────────────
PROMPT_TEXT    = 'नमस्कार। मैं हूँ आपका storyteller. आज की कहानी शुरू होती है एक ऐसी जगह से — जहाँ राज़ छुपे हैं, जहाँ सच और झूठ के बीच सिर्फ एक पतली सी लकीर है। सुनिए ध्यान से। क्योंकि यह कहानी सिर्फ सुनने की नहीं — महसूस करने की है.'

# ── GENERATION PARAMS ─────────────────────────────────────────
TEMPERATURE    = 0.7            # 0.1=robotic  1.0=natural  2.0=creative
TOP_P          = 0.8            # Nucleus sampling (0.1–1.0)
TOP_K          = 50             # Top-K vocab filter
MAX_NEW_TOKENS = 4096           # Raise for longer texts. 0 = model decides.
COMPILE        = False          # Keep False on L4 (saves VRAM)

# ── LONG TEXT CHUNKING ─────────────────────────────────────────
CHUNK_SIZE     = 200            # Words per chunk (split at sentence boundaries)
CROSSFADE_MS   = 50             # Crossfade overlap between chunks (ms)
# ──────────────────────────────────────────────────────────────

# Auto-detect device
if DEVICE == 'auto':
    try:
        import torch
        if torch.cuda.is_available():
            DEVICE = 'cuda'
            print(f"   🎯 Auto-detected: CUDA GPU → DEVICE = 'cuda'")
        else:
            DEVICE = 'cpu'
            print(f"   🎯 Auto-detected: No GPU → DEVICE = 'cpu'  (v5e-1 / T4 host RAM mode)")
    except ImportError:
        DEVICE = 'cpu'
        print(f"   🎯 Auto-detected: torch not yet installed → DEVICE = 'cpu'")
else:
    print(f"   🎯 DEVICE forced to: '{DEVICE}'")

os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs('/content/uploads', exist_ok=True)

if HF_TOKEN.strip():
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    os.environ['HF_TOKEN']              = HF_TOKEN
    print(f"   ✅ HF_TOKEN applied  (length: {len(HF_TOKEN)})")
else:
    print(f"   ℹ️  No HF_TOKEN — public model access (usually fine)")

print(f"\n{'─'*40}")
print(f"   DO_INSTALL     : {DO_INSTALL}")
print(f"   MODEL_DIR      : {MODEL_DIR}/")
print(f"   OUTPUT_DIR     : {OUTPUT_DIR}/")
print(f"   DEVICE         : {DEVICE}")
print(f"   TEMPERATURE    : {TEMPERATURE}")
print(f"   TOP_P / TOP_K  : {TOP_P} / {TOP_K}")
print(f"   MAX_NEW_TOKENS : {MAX_NEW_TOKENS}  (0 = auto)")
print(f"   COMPILE        : {COMPILE}")
print(f"   CHUNK_SIZE     : {CHUNK_SIZE} words")
print(f"   CROSSFADE_MS   : {CROSSFADE_MS} ms")
print(f"   PROMPT_TEXT    : {PROMPT_TEXT[:80]}...")
print(f"{'─'*40}")

# Initialise so later cells never hit NameError even if skipped
TEXT_TO_SYNTH   = "नमस्ते! यह FishAudio S2-Pro का परीक्षण है।"  # fallback
REFERENCE_AUDIO = ''   # ← will be set by Cell 4; empty = random voice

print(f"\n   TEXT_TO_SYNTH   : set to fallback (overwritten by Cell 3)")
print(f"   REFERENCE_AUDIO : '' (overwritten by Cell 4 if you upload audio)")

print(f"\n{SEP}")
print("  ✅ CELL 2 COMPLETE — Proceed to Cell 3")
print(f"{SEP}")


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — UPLOAD TEXT FILE  (.txt)
# ════════════════════════════════════════════════════════════
import os
from google.colab import files

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 3 — UPLOAD TEXT FILE")
print(f"{SEP}")
print("""
  Accepted : .txt  (UTF-8 encoding)
  Emotion tags supported:
    [excited]  [whisper]  [pause]   [laugh]   [sad]
    [singing]  [shouting] [emphasis][sigh]    [fast pace]
    [chuckle]  [inhale]   [volume up/down]
  Example:
    [excited] नमस्ते भाई! [pause] आज बहुत मज़ा आएगा।

  Long text? No problem! Auto-chunking handles it.
""")

print("📂 File picker opening — select your .txt file...")
uploaded_text = files.upload()

if uploaded_text:
    fname = list(uploaded_text.keys())[0]
    raw   = uploaded_text[fname]

    dest = os.path.join('/content/uploads', fname)
    with open(dest, 'wb') as f:
        f.write(raw)

    try:
        TEXT_TO_SYNTH = raw.decode('utf-8').strip()
    except UnicodeDecodeError:
        TEXT_TO_SYNTH = raw.decode('utf-8', errors='replace').strip()

    word_count = len(TEXT_TO_SYNTH.split())
    char_count = len(TEXT_TO_SYNTH)
    line_count = TEXT_TO_SYNTH.count('\n') + 1

    print(f"\n✅ FILE LOADED")
    print(f"{'─'*50}")
    print(f"   Filename   : {fname}")
    print(f"   Saved to   : {dest}")
    print(f"   Bytes      : {len(raw):,}")
    print(f"   Characters : {char_count:,}")
    print(f"   Words      : {word_count:,}")
    print(f"   Lines      : {line_count:,}")
    print(f"{'─'*50}")

    est_chunks = max(1, word_count // CHUNK_SIZE + (1 if word_count % CHUNK_SIZE else 0))
    est_min_gpu = word_count / 300  # GPU ~300 words/min
    print(f"   📊 Estimated chunks : {est_chunks} (at {CHUNK_SIZE} words/chunk)")
    print(f"   ⏱  Estimated time   : ~{est_min_gpu:.1f} min on L4 GPU")

    print(f"\n📖 PREVIEW (first 500 chars):")
    print(f"{'─'*50}")
    print(TEXT_TO_SYNTH[:500] + ('...' if char_count > 500 else ''))
    print(f"{'─'*50}")
else:
    TEXT_TO_SYNTH = "नमस्ते! यह FishAudio S2-Pro का परीक्षण है। कृपया इसे हिंदी में बोलिए।"
    print("\n⚠️  No file uploaded — using default Hindi text.")
    print(f"   Text: {TEXT_TO_SYNTH}")

print(f"\n   TEXT_TO_SYNTH set → {len(TEXT_TO_SYNTH)} chars / {len(TEXT_TO_SYNTH.split())} words")
print(f"\n{SEP}")
print("  ✅ CELL 3 COMPLETE — Proceed to Cell 4")
print(f"{SEP}")


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — UPLOAD REFERENCE AUDIO  (OPTIONAL — voice cloning)
# ════════════════════════════════════════════════════════════
import os, json
from google.colab import files

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 4 — UPLOAD REFERENCE AUDIO  (OPTIONAL)")
print(f"{SEP}")
print("""
  ┌─────────────────────────────────────────────────────┐
  │  OPTIONAL — skip this if you want a random voice.  │
  │  Click Cancel or don't upload to use random voice. │
  └─────────────────────────────────────────────────────┘

  If uploading audio:
    1. Upload .wav or .mp3 here
    2. Set PROMPT_TEXT in Cell 2 = exact transcript of clip

  Without PROMPT_TEXT, voice cloning quality drops.
""")

REFERENCE_AUDIO      = ''
REFERENCE_AUDIO_NAME = ''

print("📂 File picker — upload reference audio or cancel to skip...")
try:
    uploaded_audio = files.upload()
except Exception:
    uploaded_audio = {}

if uploaded_audio:
    aname = list(uploaded_audio.keys())[0]
    raw   = uploaded_audio[aname]
    ext   = os.path.splitext(aname)[1].lower()

    SUPPORTED = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac')
    if ext not in SUPPORTED:
        print(f"\n   ⚠️  Extension '{ext}' not in {SUPPORTED}")
        print(f"   Continuing anyway — ffmpeg may still handle it.")

    _uploads_dir = '/content/uploads'
    os.makedirs(_uploads_dir, exist_ok=True)
    dest = os.path.join(_uploads_dir, aname)
    with open(dest, 'wb') as f:
        f.write(raw)

    REFERENCE_AUDIO      = dest
    REFERENCE_AUDIO_NAME = aname
    size_kb = len(raw) / 1024

    print(f"\n✅ REFERENCE AUDIO LOADED")
    print(f"{'─'*50}")
    print(f"   Filename    : {aname}")
    print(f"   Saved to    : {dest}")
    print(f"   Size        : {size_kb:.1f} KB  ({len(raw):,} bytes)")
    print(f"   Format      : {ext}")
    print(f"{'─'*50}")

    try:
        import subprocess as _sp
        r = _sp.run(
            ['ffprobe','-v','quiet','-print_format','json','-show_streams', dest],
            capture_output=True, text=True, timeout=10
        )
        probe = json.loads(r.stdout)
        st = probe['streams'][0]
        duration    = float(st.get('duration', 0))
        sample_rate = st.get('sample_rate', '?')
        channels_n  = st.get('channels', '?')
        codec       = st.get('codec_name', '?')
        print(f"   Duration    : {duration:.2f} s")
        print(f"   Sample rate : {sample_rate} Hz")
        print(f"   Channels    : {channels_n}")
        print(f"   Codec       : {codec}")
        if duration < 2:
            print(f"   ⚠️  Very short (<2s) — quality may suffer.")
        elif duration > 30:
            print(f"   ⚠️  Long clip (>30s) — first ~15s will be used.")
        else:
            print(f"   ✅ Good clip length for voice cloning.")
    except Exception as e:
        print(f"   (Could not probe audio: {e})")

    if not PROMPT_TEXT.strip():
        print(f"\n   ⚠️  PROMPT_TEXT is empty in Cell 2!")
        print(f"   Voice cloning quality is much better with a transcript.")
        print(f"   → Go back to Cell 2 and set PROMPT_TEXT to what is spoken.")
    else:
        excerpt = PROMPT_TEXT[:80] + ('...' if len(PROMPT_TEXT) > 80 else '')
        print(f"\n   PROMPT_TEXT : '{excerpt}'")
        print(f"   ✅ Transcript provided — full voice cloning enabled.")

else:
    REFERENCE_AUDIO = ''
    print("\n✅ No reference audio — will use random built-in voice.")
    print("   S2-Pro's default voices are expressive and multilingual.")

print(f"\n{'─'*50}")
print(f"   REFERENCE_AUDIO = '{REFERENCE_AUDIO}'")
mode = f"VOICE CLONE: {REFERENCE_AUDIO_NAME}" if REFERENCE_AUDIO else "RANDOM VOICE (no reference)"
print(f"   Mode            = {mode}")
print(f"{'─'*50}")

print(f"\n{SEP}")
print("  ✅ CELL 4 COMPLETE — Proceed to Cell 5")
print(f"{SEP}")


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — INSTALL DEPENDENCIES  (fish-speech + ecosystem)
# ════════════════════════════════════════════════════════════
# Safe for Colab L4 GPU — keeps pre-installed CUDA PyTorch.
# ════════════════════════════════════════════════════════════
import os, sys, subprocess, time

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 5 — INSTALLING DEPENDENCIES")
print(f"{SEP}\n")

if not DO_INSTALL:
    print("   ⏭️  DO_INSTALL = False — skipping.\n")
    print(f"{SEP}")
    print("  ✅ CELL 5 SKIPPED — Proceed to Cell 6")
    print(f"{SEP}")
else:
    def banner(msg):
        print(f"\n{'─'*60}")
        print(f"  {msg}")
        print(f"{'─'*60}")

    def pip_install(label, pkgs, no_deps=False, allow_fail=False):
        if isinstance(pkgs, str):
            pkgs = [pkgs]
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
        if no_deps:
            cmd.insert(5, '--no-deps')
        print(f"   📦 {label} ...", end='', flush=True)
        t0 = time.time()
        r = subprocess.run(cmd, capture_output=True, text=True)
        elapsed = time.time() - t0
        if r.returncode == 0:
            print(f"  ✅  ({elapsed:.1f}s)")
        else:
            if allow_fail:
                last = [l for l in r.stderr.strip().splitlines() if l.strip()]
                msg = last[-1][:100] if last else '(no stderr)'
                print(f"  ⚠️  non-fatal ({elapsed:.1f}s)\n     {msg}")
            else:
                print(f"  ❌  ({elapsed:.1f}s)")
                print(r.stderr.strip()[-400:])
                raise RuntimeError(f"pip install failed: {label}")
        return r.returncode == 0

    grand_t0 = time.time()

    # ── 1: System packages ───────────────────────────────────────
    banner("1 / 6 — System packages (apt)")
    subprocess.run(['apt-get','update','-qq'], capture_output=True)
    subprocess.run(['apt-get','install','-y','-qq',
                    'ffmpeg','libsox-dev','sox','libsndfile1-dev',
                    'libasound2-dev','cmake','build-essential','git'],
                   capture_output=True)
    print("   ✅  apt packages installed")

    # ── 2: Pip upgrade ───────────────────────────────────────────
    banner("2 / 6 — Upgrade pip, setuptools, wheel")
    pip_install("pip upgrade", ['pip','setuptools','wheel','--upgrade'])

    # ── 3: PyTorch ───────────────────────────────────────────────
    banner("3 / 6 — PyTorch (using Colab pre-installed CUDA build)")
    import torch
    torch.set_num_threads(8)
    print(f"   ✅  Using Colab torch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  threads: {torch.get_num_threads()}")
    print(f"   ℹ️  Keeping Colab's CUDA PyTorch — needed for GPU inference")

    # ── 4: transformers FIRST ────────────────────────────────────
    banner("4 / 6 — transformers + core packages")
    for pkg, lbl in [
        ('transformers>=4.45.2,<=4.57.3',  'transformers (pins hf_hub<1.0)'),
        ('accelerate>=0.26.0',             'accelerate'),
        ('vector_quantize_pytorch==1.14.24','vector_quantize_pytorch'),
        ('numpy>=1.26.0',                  'numpy'),
        ('scipy',                          'scipy'),
        ('librosa>=0.10.1',                'librosa'),
        ('soundfile',                      'soundfile'),
    ]:
        pip_install(lbl, pkg)

    # ── 5: fish-speech specific ──────────────────────────────────
    banner("5 / 6 — fish-speech specific packages")
    FISH_PKGS = [
        ('loguru>=0.6.0',               'loguru'),
        ('einops>=0.7.0',               'einops'),
        ('loralib>=0.1.2',              'loralib'),
        ('pyrootutils>=1.0.4',          'pyrootutils'),
        ('hydra-core>=1.3.2',           'hydra-core'),
        ('lightning>=2.1.0',            'lightning'),
        ('natsort>=8.4.0',              'natsort'),
        ('rich>=13.5.3',                'rich'),
        ('tiktoken>=0.8.0',             'tiktoken'),
        ('pydantic>=2.9.2',             'pydantic'),
        ('zstandard>=0.22.0',           'zstandard'),
        ('safetensors',                 'safetensors'),
        ('einx[torch]==0.2.2',          'einx'),
        ('resampy>=0.4.3',              'resampy'),
        ('pydub',                       'pydub'),
        ('tqdm',                        'tqdm'),
        ('silero-vad',                  'silero-vad'),
        ('opencc-python-reimplemented==0.1.7', 'opencc'),
        ('datasets==2.18.0',            'datasets'),
        ('rotary-embedding-torch',      'rotary-embedding-torch'),
        ('descript-audio-codec',        'descript-audio-codec'),
    ]
    for pkg, lbl in FISH_PKGS:
        pip_install(lbl, pkg, allow_fail=True)

    # ── 6: fish-speech repo ──────────────────────────────────────
    banner("6 / 6 — fish-speech GitHub repo + editable install")
    REPO_DIR = '/content/fish-speech'
    if not os.path.exists(os.path.join(REPO_DIR, '.git')):
        print("   Cloning fishaudio/fish-speech...", end='', flush=True)
        t0 = time.time()
        r = subprocess.run(
            ['git','clone','--depth','1',
             'https://github.com/fishaudio/fish-speech.git', REPO_DIR],
            capture_output=True, text=True)
        if r.returncode == 0:
            print(f"  ✅  ({time.time()-t0:.0f}s)")
        else:
            print(f"  ❌\n{r.stderr[-200:]}")
    else:
        print(f"   ✅  {REPO_DIR} already exists")

    if os.path.exists(REPO_DIR):
        pip_install("fish-speech editable", ['-e', REPO_DIR, '--no-deps'],
                    no_deps=False, allow_fail=True)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        print(f"   ✅  {REPO_DIR} added to sys.path")

    # ── Verify ───────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("  🔍 IMPORT VERIFICATION")
    print(f"{'─'*60}")
    VERIFY = [
        ('torch',                   'import torch; print(torch.__version__)'),
        ('torchaudio',              'import torchaudio; print(torchaudio.__version__)'),
        ('transformers',            'import transformers; print(transformers.__version__)'),
        ('huggingface_hub',         'import huggingface_hub; print(huggingface_hub.__version__)'),
        ('vector_quantize_pytorch', 'import vector_quantize_pytorch; print("ok")'),
        ('loguru',                  'import loguru; print(loguru.__version__)'),
        ('soundfile',               'import soundfile; print(soundfile.__version__)'),
        ('tiktoken',                'import tiktoken; print(tiktoken.__version__)'),
        ('loralib',                 'import loralib; print("ok")'),
        ('fish_speech',             'import fish_speech; print("ok")'),
    ]
    passed, failed = [], []
    for name, code in VERIFY:
        r = subprocess.run([sys.executable,'-c',code], capture_output=True, text=True)
        if r.returncode == 0:
            ver = r.stdout.strip().splitlines()[-1]
            print(f"   ✅  {name:35s} {ver}")
            passed.append(name)
        else:
            err = (r.stderr + r.stdout).strip()[-80:]
            print(f"   ❌  {name:35s} {err}")
            failed.append(name)

    total_t = time.time() - grand_t0
    print(f"\n   Total time : {total_t:.0f}s  ({total_t/60:.1f} min)")
    print(f"   Passed     : {len(passed)}/{len(VERIFY)}")
    if failed:
        print(f"   Failed     : {failed}")
        print(f"\n   ⚠️  Try: Runtime → Restart session, then re-run Cell 5.")
    else:
        print(f"   ✅ All imports verified!")

    print(f"\n{SEP}")
    print("  ✅ CELL 5 COMPLETE — Proceed to Cell 6")
    print(f"  ⚠️  If any import failed: Runtime → Restart → re-run Cell 5")
    print(f"{SEP}")


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — DOWNLOAD MODEL WEIGHTS  (fishaudio/s2-pro)
# ════════════════════════════════════════════════════════════
import os, sys, time

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 6 — DOWNLOADING MODEL WEIGHTS")
print(f"{SEP}")
print(f"""
   Model      : fishaudio/s2-pro
   Destination: {MODEL_DIR}
   Size       : ~10 GB  (first run)
   Resume     : ✅ existing files are skipped automatically
   Method     : huggingface_hub Python API  (not CLI)
""")

try:
    import huggingface_hub as hfhub
    print(f"   huggingface_hub : {hfhub.__version__} ✅")
except ImportError:
    print("   huggingface_hub not found — installing...")
    import subprocess
    subprocess.run([sys.executable,'-m','pip','install','huggingface_hub','-q'],
                   capture_output=True)
    import huggingface_hub as hfhub

from huggingface_hub import snapshot_download, hf_hub_download

# ── Show existing files ─────────────────────────────────────
print(f"\n🔍 Scanning {MODEL_DIR} ...")
existing = []
if os.path.exists(MODEL_DIR):
    for root, dirs, fnames in os.walk(MODEL_DIR):
        for fn in sorted(fnames):
            fp  = os.path.join(root, fn)
            rel = os.path.relpath(fp, MODEL_DIR)
            sz  = os.path.getsize(fp)
            existing.append((rel, sz))

if existing:
    total_cached = sum(s for _, s in existing)
    print(f"   Found {len(existing)} cached file(s)  ({total_cached/1e9:.2f} GB total):")
    for rel, sz in sorted(existing):
        bar_len = int(sz / 1e8)
        bar = '█' * min(bar_len, 20)
        print(f"   📄 {rel:50s} {sz/1e6:8.1f} MB  {bar}")
    REQUIRED = ['config.json', 'codec.pth']
    have_req  = all(any(r.endswith(req) for r, _ in existing) for req in REQUIRED)
    if have_req:
        print(f"\n   ✅ Critical files already cached.")
    else:
        missing = [r for r in REQUIRED if not any(e.endswith(r) for e, _ in existing)]
        print(f"\n   ⚠️  Missing critical files: {missing}")
else:
    print(f"   📭 Empty — full download needed (~10 GB, 5–25 min)")

# ── Download ────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  📥 Running snapshot_download() ...")
print(f"{'─'*60}\n")

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

t0 = time.time()
try:
    local_path = snapshot_download(
        repo_id                = 'fishaudio/s2-pro',
        repo_type              = 'model',
        local_dir              = MODEL_DIR,
        local_dir_use_symlinks = False,
        token                  = HF_TOKEN.strip() or None,
        ignore_patterns        = ['*.msgpack','flax_model*','tf_model*','rust_model*'],
    )
    elapsed = time.time() - t0
    print(f"\n   ✅ snapshot_download complete  ({elapsed/60:.1f} min)")
    print(f"   Files at: {local_path}")

except Exception as e:
    print(f"\n   ⚠️  snapshot_download raised {type(e).__name__}: {e}")
    print("   Trying file-by-file fallback ...")
    CRITICAL = ['config.json','codec.pth','model.pth',
                'tokenizer.json','tokenizer_config.json']
    any_ok = False
    for fname in CRITICAL:
        dest = os.path.join(MODEL_DIR, fname)
        if os.path.exists(dest):
            print(f"   ✅ {fname} — already present, skip")
            continue
        try:
            print(f"   📥 {fname} ...", end='', flush=True)
            hf_hub_download(repo_id='fishaudio/s2-pro', filename=fname,
                            local_dir=MODEL_DIR, local_dir_use_symlinks=False,
                            token=HF_TOKEN.strip() or None)
            sz = os.path.getsize(dest) / 1e6
            print(f" ✅  ({sz:.1f} MB)")
            any_ok = True
        except Exception as fe:
            print(f" ⚠️  {type(fe).__name__}")
    if not any_ok:
        raise RuntimeError("All download attempts failed. Check network / HF status.")

# ── Final inventory ─────────────────────────────────────────
print(f"\n{'─'*60}")
print("  🔍 FINAL INVENTORY")
print(f"{'─'*60}")
total_gb = 0.0
all_files = []
for root, dirs, fnames in os.walk(MODEL_DIR):
    for fn in sorted(fnames):
        fp  = os.path.join(root, fn)
        rel = os.path.relpath(fp, MODEL_DIR)
        sz  = os.path.getsize(fp)
        total_gb += sz / 1e9
        all_files.append((rel, sz))
        print(f"   📄 {rel:50s} {sz/1e6:8.1f} MB")
print(f"   {'─'*55}")
print(f"   {'Total:':50s} {total_gb:8.2f} GB")

REQUIRED = ['config.json', 'codec.pth']
print(f"\n  🔍 CRITICAL FILE CHECK")
all_ok = True
for req in REQUIRED:
    found = any(r.endswith(req) for r, _ in all_files)
    print(f"   {'✅' if found else '❌ MISSING'}  {req}")
    if not found: all_ok = False

if not all_ok:
    raise RuntimeError("Critical files missing. Re-run this cell to resume download.")

print(f"\n{SEP}")
print("  ✅ CELL 6 COMPLETE — Proceed to Cell 7")
print(f"{SEP}")


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — S2-PRO INFERENCE  (Chunked GPU Pipeline)
# ════════════════════════════════════════════════════════════
# Stages per chunk:
#   1. Ref-audio → DAC-encode prompt tokens  (once, reused)
#   2. Text → semantic tokens  (4B LLM on GPU)
#   3. Semantic tokens → WAV   (DAC decoder on GPU)
# Long text is auto-chunked at sentence boundaries.
# All chunks share the same reference audio for consistent voice.
# ════════════════════════════════════════════════════════════
import os, sys, time, gc, re, glob, shlex, subprocess
import html as _html
import numpy as np
from IPython.display import display, HTML, clear_output

REPO_DIR = '/content/fish-speech'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 7 — S2-PRO INFERENCE  (L4 GPU · CHUNKED)")
print(f"{SEP}\n")
print(f"   TEXT_TO_SYNTH   : {len(TEXT_TO_SYNTH)} chars / {len(TEXT_TO_SYNTH.split())} words")
print(f"   REFERENCE_AUDIO : '{REFERENCE_AUDIO}'" if REFERENCE_AUDIO
      else "   REFERENCE_AUDIO : '' (random voice)")
print(f"   MODEL_DIR       : {MODEL_DIR}")
print(f"   OUTPUT_DIR      : {OUTPUT_DIR}")
print(f"   DEVICE          : {DEVICE}")
print(f"   TEMPERATURE     : {TEMPERATURE}")
print(f"   TOP_P / TOP_K   : {TOP_P} / {TOP_K}")
print(f"   CHUNK_SIZE      : {CHUNK_SIZE} words")
print(f"   CROSSFADE_MS    : {CROSSFADE_MS} ms")
print(f"   COMPILE         : {COMPILE}")

for req in ['config.json', 'codec.pth']:
    rp = os.path.join(MODEL_DIR, req)
    if not os.path.exists(rp):
        raise FileNotFoundError(f"Missing: {rp} — re-run Cell 6.")
    print(f"   ✅ {req}")

if not TEXT_TO_SYNTH.strip():
    raise ValueError("TEXT_TO_SYNTH is empty — run Cell 3 first.")
print(f"   ✅ Input text present")

try:
    with open('/proc/meminfo') as f:
        mi = {l.split(':')[0]: l.split(':')[1].strip() for l in f}
    _avail_gb = int(mi['MemAvailable'].split()[0]) / 1e6
    _total_gb = int(mi['MemTotal'].split()[0])     / 1e6
    print(f"   RAM available   : {_avail_gb:.1f} GB / {_total_gb:.1f} GB")
except Exception:
    _total_gb = 0

# Clean old outputs
for pattern in ['codes_*.npy', 'fake.npy', 'fake.wav', 'ref_prompt.npy']:
    for f in glob.glob(os.path.join(REPO_DIR, pattern)):
        os.remove(f)
        print(f"   🧹 Removed: {os.path.basename(f)}")
for f in glob.glob(os.path.join(OUTPUT_DIR, 'chunk_*.wav')):
    os.remove(f)

gc.collect()
print()

# ════════════════════════════════════════════════════════════
# TEXT CHUNKING (sentence-boundary aware)
# ════════════════════════════════════════════════════════════
def chunk_text(text, max_words=200):
    """Split text into chunks at sentence boundaries, respecting max_words."""
    # Sentence-ending patterns: Hindi purna viram, period, !, ?, newline
    sentences = re.split(r'(?<=[।.!?\n])\s*', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current = []
    current_wc = 0

    for sent in sentences:
        sent_wc = len(sent.split())
        # If single sentence exceeds max_words, force-split by words
        if sent_wc > max_words:
            if current:
                chunks.append(' '.join(current))
                current = []
                current_wc = 0
            words = sent.split()
            for i in range(0, len(words), max_words):
                chunks.append(' '.join(words[i:i+max_words]))
            continue

        if current_wc + sent_wc > max_words and current:
            chunks.append(' '.join(current))
            current = [sent]
            current_wc = sent_wc
        else:
            current.append(sent)
            current_wc += sent_wc

    if current:
        chunks.append(' '.join(current))

    return chunks if chunks else [text.strip()]

text_clean = TEXT_TO_SYNTH.replace('\r\n', '\n').strip()
total_words = len(text_clean.split())

if total_words <= CHUNK_SIZE * 1.3:
    # Short text — single chunk, no splitting
    text_chunks = [text_clean.replace('\n', ' ').strip()]
else:
    text_chunks = chunk_text(text_clean, max_words=CHUNK_SIZE)

num_chunks = len(text_chunks)
print(f"📄 Text split into {num_chunks} chunk(s)  ({total_words} total words)")
for ci, ch in enumerate(text_chunks):
    wc = len(ch.split())
    preview = ch[:60].replace('\n',' ') + ('...' if len(ch) > 60 else '')
    print(f"   Chunk {ci+1}: {wc} words — {preview}")
print()

# ════════════════════════════════════════════════════════════
# DASHBOARD RENDERER
# ════════════════════════════════════════════════════════════
_dashboard_handle = display(HTML(""), display_id=True)

def _get_ram():
    try:
        with open('/proc/meminfo') as f:
            mi = {l.split(':')[0]: l.split(':')[1].strip() for l in f}
        used  = (int(mi['MemTotal'].split()[0]) - int(mi['MemAvailable'].split()[0])) / 1e6
        total = int(mi['MemTotal'].split()[0]) / 1e6
        return used, total
    except Exception:
        return 0, _total_gb or 96

def _get_gpu_mem():
    try:
        r = subprocess.run(['nvidia-smi','--query-gpu=memory.used,memory.total',
                            '--format=csv,noheader,nounits'],
                           capture_output=True, text=True, timeout=3)
        parts = r.stdout.strip().split(',')
        return float(parts[0].strip())/1024, float(parts[1].strip())/1024
    except Exception:
        return 0, 0

def _bar(pct, w=26, fill='█', empty='░', color='#00d4ff'):
    n = int(w * max(0, min(100, pct)) / 100)
    b = fill * n + empty * (w - n)
    return (
        '<span style="color:' + color +
        ';font-family:monospace;letter-spacing:1px">' + b + '</span>'
    )

def render_dashboard(stage, stage_label, log_lines, elapsed,
                     chunk_idx=0, total_chunks=1,
                     tok_done=0, tok_est=0, stage_pct=0, overall_pct=0,
                     status='RUNNING', error_msg=None,
                     cur_tps=0.0, cur_eta=0.0, chunk_eta=0.0):
    ram_used, ram_total = _get_ram()
    ram_pct  = (ram_used / ram_total * 100) if ram_total else 0
    gpu_used, gpu_total = _get_gpu_mem()
    gpu_pct = (gpu_used / gpu_total * 100) if gpu_total else 0
    mins, secs = divmod(int(elapsed), 60)
    hrs, mins = divmod(mins, 60)
    if hrs > 0:
        elapsed_str = f"{hrs:d}:{mins:02d}:{secs:02d}"
    else:
        elapsed_str = f"{mins:02d}:{secs:02d}"

    # Speed / ETA
    if cur_tps > 0:
        tps_str = f"{cur_tps:.1f} tok/s"
        if cur_eta > 0:
            em, es = divmod(int(cur_eta), 60)
            eta_str = f"~{em:02d}:{es:02d}"
        elif tok_est > 0 and tok_done >= tok_est:
            eta_str = "almost done"
        else:
            eta_str = "─"
    elif tok_done > 0 and elapsed > 2 and tok_est > tok_done:
        tps = tok_done / elapsed
        eta_sec = (tok_est - tok_done) / tps
        tps_str = f"{tps:.1f} tok/s"
        em, es = divmod(int(eta_sec), 60)
        eta_str = f"~{em:02d}:{es:02d}"
    elif tok_done > 0 and elapsed > 2:
        tps = tok_done / elapsed
        tps_str = f"{tps:.1f} tok/s"
        eta_str = "almost done"
    else:
        tps_str = eta_str = "─"

    # Overall ETA from chunk progress
    if chunk_eta > 0:
        oem, oes = divmod(int(chunk_eta), 60)
        overall_eta_str = f"~{oem:02d}:{oes:02d}"
    else:
        overall_eta_str = eta_str

    sc = {'RUNNING': '#00d4ff', 'DONE': '#7fff00',
          'ERROR': '#ff4444', 'WAITING': '#ffa500'}
    status_color = sc.get(status, '#00d4ff')

    log_html = ""
    for line in log_lines[-9:]:
        safe = _html.escape(str(line))
        lo = line.lower()
        if any(w in lo for w in ['error', 'exception', 'failed', 'traceback']):
            c = '#ff6868'
        elif any(w in lo for w in ['info', 'load', 'done', 'complete', 'saved', '✅']):
            c = '#88ff88'
        elif any(w in lo for w in ['warn', 'warning', 'skip']):
            c = '#ffbb44'
        elif line.startswith('[') and ']' in line:
            c = '#aaddff'
        else:
            c = '#7aa8cc'
        log_html += (
            '<div style="color:' + c +
            ';margin:1px 0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
            '&gt; ' + safe + '</div>'
        )

    err_block = ""
    if error_msg:
        err_block = (
            '<div style="margin-top:8px;padding:6px 10px;background:#3a0000;'
            'border:1px solid #ff4444;border-radius:4px">'
            '<b style="color:#ff6666">⚠ ERROR:</b>'
            '<span style="color:#ffaaaa"> ' +
            _html.escape(str(error_msg)[:220]) +
            '</span></div>'
        )

    chunk_str = f"Chunk {chunk_idx}/{total_chunks}" if total_chunks > 1 else "Single"
    tok_disp   = f"{tok_done:,}" if tok_done else "─"
    total_disp = f"~{tok_est:,}" if tok_est  else "─"
    sp_str  = f"{stage_pct:.1f}%"
    op_str  = f"{overall_pct:.1f}%"
    ru_str  = f"{ram_used:.1f}/{ram_total:.0f} GB"
    gu_str  = f"{gpu_used:.1f}/{gpu_total:.0f} GB" if gpu_total else "─"

    html = (
        '<div style="background:#080c18;border:2px solid #00d4ff;border-radius:10px;'
        'padding:18px 22px;font-family:\'Courier New\',monospace;'
        'box-shadow:0 0 30px rgba(0,212,255,0.25);margin:12px 0;max-width:820px">'

        # header
        '<div style="display:flex;justify-content:space-between;align-items:flex-start;'
        'border-bottom:1px solid #162840;padding-bottom:10px;margin-bottom:14px">'
        '<div>'
        '<div style="color:#00d4ff;font-size:17px;font-weight:bold;letter-spacing:2px">'
        '🤖 S2-PRO SYNTHESIS ENGINE</div>'
        '<div style="color:#3a6080;font-size:11px;margin-top:3px">'
        'fishaudio/s2-pro &nbsp;·&nbsp; ' + ('CUDA GPU' if DEVICE == 'cuda' else 'CPU HOST RAM') + ' &nbsp;·&nbsp; ' + chunk_str +
        '</div></div>'
        '<div style="text-align:right">'
        '<div style="color:' + status_color + ';font-weight:bold;font-size:14px;'
        'text-shadow:0 0 10px ' + status_color + '">● ' + status + '</div>'
        '<div style="color:#3a6080;font-size:11px;margin-top:3px">⏱ ' + elapsed_str + '</div>'
        '</div></div>'

        # stage bar
        '<div style="margin-bottom:14px">'
        '<div style="color:#3a6080;font-size:10px;letter-spacing:2px;margin-bottom:5px">'
        'STAGE ' + str(stage) + ' / 3</div>'
        '<div style="color:#e8f4ff;font-size:13px;font-weight:bold;margin-bottom:7px">'
        + _html.escape(str(stage_label)) +
        '</div>'
        '<div style="display:flex;align-items:center;gap:12px">'
        + _bar(stage_pct, w=30, color='#00d4ff') +
        '<span style="color:#00d4ff;font-size:13px;min-width:48px">' + sp_str + '</span>'
        '</div></div>'

        # metrics grid
        '<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:10px 18px;'
        'margin-bottom:14px;font-size:12px">'
        '<div><span style="color:#3a6080">TOKENS</span><br>'
        '<span style="color:#e8f4ff">' + tok_disp + ' / ' + total_disp + '</span></div>'
        '<div><span style="color:#3a6080">SPEED</span><br>'
        '<span style="color:#e8f4ff">' + tps_str + '</span></div>'
        '<div><span style="color:#3a6080">STAGE ETA</span><br>'
        '<span style="color:#e8f4ff">' + eta_str + '</span></div>'
        '<div><span style="color:#3a6080">RAM</span><br>'
        '<span style="color:#e8f4ff">' + ru_str + '</span></div>'
        '<div><span style="color:#3a6080">GPU VRAM</span><br>'
        '<span style="color:#e8f4ff">' + gu_str + '</span></div>'
        '<div><span style="color:#3a6080">OVERALL ETA</span><br>'
        '<span style="color:#e8f4ff">' + overall_eta_str + '</span></div>'
        '</div>'

        # overall bar
        '<div style="margin-bottom:12px">'
        '<div style="color:#3a6080;font-size:10px;letter-spacing:2px;margin-bottom:5px">'
        'OVERALL PROGRESS</div>'
        '<div style="display:flex;align-items:center;gap:12px">'
        + _bar(overall_pct, w=40, color='#7fff00') +
        '<span style="color:#7fff00;font-size:13px;font-weight:bold;min-width:48px">'
        + op_str + '</span>'
        '</div></div>'

        # log
        '<div style="background:#060a14;border:1px solid #162840;border-radius:5px;'
        'padding:8px 12px;font-size:11px;max-height:180px;overflow-y:auto">'
        + log_html + '</div>'
        + err_block +
        '</div>'
    )
    _dashboard_handle.update(HTML(html))

def upd(**kw):
    render_dashboard(**kw)

# ════════════════════════════════════════════════════════════
# SUBPROCESS RUNNER WITH TQDM PARSING
# ════════════════════════════════════════════════════════════
def run_stage(cmd, stage_num, stage_label, log_lines, t0,
              pct_start, pct_end, est_tokens, chunk_idx=0,
              total_chunks=1, overall_base=0, overall_span=100,
              chunk_eta_val=0.0):
    """Run a subprocess, parse tqdm progress, update dashboard."""
    log_lines.append(f"[{time.strftime('%H:%M:%S')}] Stage {stage_num}: {stage_label}")
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'

    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True, env=env
    )

    tok_done = 0
    stage_t0 = time.time()
    last_tps = 0.0
    last_eta = 0.0
    tqdm_re = re.compile(r'(\d+)[/|](\d+)\s*\[.*?(\d+\.\d+)\s*(it|tok)/s')
    pct_re  = re.compile(r'(\d+)%\|')

    for raw_line in proc.stdout:
        line = raw_line.rstrip()
        if not line:
            continue

        # Parse tqdm-style output
        m = tqdm_re.search(line)
        if m:
            tok_done = int(m.group(1))
            tok_total = int(m.group(2))
            last_tps  = float(m.group(3))
            if tok_total > tok_done and last_tps > 0:
                last_eta = (tok_total - tok_done) / last_tps
            else:
                last_eta = 0
            if tok_total > 0:
                frac = tok_done / tok_total
            else:
                frac = 0
            s_pct = frac * 100
            o_pct = overall_base + overall_span * (pct_start + (pct_end - pct_start) * frac) / 100
        else:
            pm = pct_re.search(line)
            if pm:
                s_pct = int(pm.group(1))
                frac = s_pct / 100
                o_pct = overall_base + overall_span * (pct_start + (pct_end - pct_start) * frac) / 100
            else:
                s_pct = 0
                o_pct = overall_base + overall_span * pct_start / 100

            # Log interesting lines
            lo = line.lower()
            if any(w in lo for w in ['load', 'error', 'warn', 'saved', 'done',
                                     'compil', 'generat', 'token', 'info', 'model']):
                short = line[:120]
                if short not in [l[-120:] for l in log_lines[-5:]]:
                    log_lines.append(f"[{time.strftime('%H:%M:%S')}] {short}")

        upd(stage=stage_num, stage_label=stage_label,
            log_lines=log_lines, elapsed=time.time()-t0,
            chunk_idx=chunk_idx, total_chunks=total_chunks,
            tok_done=tok_done, tok_est=est_tokens,
            stage_pct=s_pct, overall_pct=min(o_pct, 99.9),
            cur_tps=last_tps, cur_eta=last_eta, chunk_eta=chunk_eta_val)

    proc.wait()
    stage_elapsed = time.time() - stage_t0
    log_lines.append(
        f"[{time.strftime('%H:%M:%S')}] Stage {stage_num} done ({stage_elapsed:.1f}s, exit={proc.returncode})"
    )
    o_pct = overall_base + overall_span * pct_end / 100
    upd(stage=stage_num, stage_label=f"✅ {stage_label}",
        log_lines=log_lines, elapsed=time.time()-t0,
        chunk_idx=chunk_idx, total_chunks=total_chunks,
        tok_done=tok_done, tok_est=est_tokens,
        stage_pct=100, overall_pct=min(o_pct, 99.9),
        cur_tps=last_tps, cur_eta=0.0, chunk_eta=chunk_eta_val)
    return proc.returncode, stage_elapsed

# ════════════════════════════════════════════════════════════
# STAGE 1: Encode reference audio (once, reused for all chunks)
# ════════════════════════════════════════════════════════════
t0 = time.time()
log_lines = [f"[{time.strftime('%H:%M:%S')}] Inference started — {num_chunks} chunk(s) on {DEVICE.upper()}"]
prompt_npy = ''
USE_OWN_REFERENCE = bool(REFERENCE_AUDIO and os.path.exists(REFERENCE_AUDIO))

if USE_OWN_REFERENCE:
    log_lines.append(f"[{time.strftime('%H:%M:%S')}] Stage 1: Encoding reference audio")
    _stage1_label = f"Reference Audio → DAC Tokens  ({DEVICE.upper()})"
    upd(stage=1, stage_label=_stage1_label,
        log_lines=log_lines, elapsed=0,
        chunk_idx=0, total_chunks=num_chunks,
        stage_pct=0, overall_pct=0, cur_tps=0.0, cur_eta=0.0)

    # Clean any stale fake.npy / fake.wav from previous runs
    for _old in glob.glob(os.path.join(REPO_DIR, 'fake.*')):
        os.remove(_old)

    # DAC inference writes fake.npy (tokens) + fake.wav (recon) to cwd.
    # We do NOT pass -o (it's treated as a file path, not a directory).
    cmd_enc = (
        f"{sys.executable} -m fish_speech.models.dac.inference"
        f" -i {shlex.quote(REFERENCE_AUDIO)}"
        f" --checkpoint-path {shlex.quote(os.path.join(MODEL_DIR, 'codec.pth'))}"
        f" --device {DEVICE}"
    )

    rc1, _ = run_stage(cmd_enc, 1, _stage1_label,
                       log_lines, t0, pct_start=0, pct_end=100, est_tokens=200,
                       chunk_idx=0, total_chunks=num_chunks,
                       overall_base=0, overall_span=5)

    if rc1 == 0:
        # DAC writes fake.npy to the working directory (REPO_DIR)
        _ref_npy_cwd = os.path.join(REPO_DIR, 'fake.npy')
        if os.path.exists(_ref_npy_cwd):
            # Move to a safe location so chunk processing doesn't overwrite it
            prompt_npy = os.path.join(OUTPUT_DIR, 'ref_prompt.npy')
            import shutil as _shutil
            _shutil.copy2(_ref_npy_cwd, prompt_npy)
            log_lines.append(f"[{time.strftime('%H:%M:%S')}] ✅ Prompt tokens saved: ref_prompt.npy")
        else:
            # Fallback: search more broadly
            npys = sorted(glob.glob(os.path.join(REPO_DIR, '*.npy')))
            if npys:
                prompt_npy = npys[-1]
                log_lines.append(f"[{time.strftime('%H:%M:%S')}] ✅ Prompt tokens: {os.path.basename(prompt_npy)}")
            else:
                err_msg = "Stage 1 completed but no .npy output found in working directory."
                upd(stage=1, stage_label="❌ No .npy output from Stage 1",
                    log_lines=log_lines, elapsed=time.time()-t0,
                    stage_pct=100, overall_pct=5, status="ERROR",
                    error_msg=err_msg, cur_tps=0.0, cur_eta=0.0)
                raise RuntimeError(err_msg)
    else:
        err_msg = "Stage 1 FAILED — DAC could not encode reference audio."
        upd(stage=1, stage_label="❌ Stage 1 FAILED",
            log_lines=log_lines, elapsed=time.time()-t0,
            stage_pct=100, overall_pct=5, status="ERROR",
            error_msg=err_msg, cur_tps=0.0, cur_eta=0.0)
        raise RuntimeError(err_msg)
else:
    log_lines.append(f"[{time.strftime('%H:%M:%S')}] Stage 1 skipped (no reference audio)")
    upd(stage=1, stage_label="Stage 1: Skipped — no reference audio",
        log_lines=log_lines, elapsed=time.time() - t0,
        chunk_idx=0, total_chunks=num_chunks,
        stage_pct=100, overall_pct=5, status='WAITING',
        cur_tps=0.0, cur_eta=0.0)
    time.sleep(0.4)

# ════════════════════════════════════════════════════════════
# PROCESS CHUNKS: Stage 2 + Stage 3 per chunk
# ════════════════════════════════════════════════════════════
chunk_wavs = []
chunk_times = []       # Track actual time per chunk for ETA
stage1_time = time.time() - t0

# Overall progress: 5% for stage 1, remaining 95% split across chunks
chunk_pct_each = 95.0 / num_chunks

for ci, chunk_text in enumerate(text_chunks):
    chunk_num = ci + 1
    chunk_t0 = time.time()
    chunk_words = len(chunk_text.split())
    est_tokens = max(chunk_words * 25, 500)

    chunk_base_pct = 5.0 + ci * chunk_pct_each  # overall % base for this chunk

    # Calculate chunk ETA from previous chunks
    if chunk_times:
        avg_chunk_time = sum(chunk_times) / len(chunk_times)
        remaining_chunks = num_chunks - ci
        chunk_eta_val = avg_chunk_time * remaining_chunks
    else:
        # Estimate: ~2 sec per word on GPU for first chunk
        chunk_eta_val = chunk_words * 0.5 * (num_chunks - ci)

    log_lines.append(
        f"[{time.strftime('%H:%M:%S')}] ── Chunk {chunk_num}/{num_chunks}: "
        f"{chunk_words} words, est ~{est_tokens} tokens ──"
    )

    # Clean previous chunk outputs (codes + fake.wav to prevent collision)
    for _pattern in ['codes_*.npy', 'fake.wav', 'fake.npy']:
        for f in glob.glob(os.path.join(REPO_DIR, _pattern)):
            # Don't delete ref_prompt.npy (our saved reference tokens)
            if os.path.basename(f) == 'ref_prompt.npy':
                continue
            os.remove(f)
    for f in glob.glob(os.path.join(OUTPUT_DIR, 'codes_*.npy')):
        os.remove(f)
    # Also clean any stale fake.wav in OUTPUT_DIR (from previous chunks)
    for f in glob.glob(os.path.join(OUTPUT_DIR, 'fake.wav')):
        os.remove(f)

    # ── Stage 2: text → semantic tokens ──────────────────────
    chunk_text_clean = chunk_text.replace('\n', ' ').strip()
    cmd_t2s = (
        f"{sys.executable} -m fish_speech.models.text2semantic.inference"
        f" --text {shlex.quote(chunk_text_clean)}"
        f" --checkpoint-path {shlex.quote(MODEL_DIR)}"
        f" --output-dir {shlex.quote(OUTPUT_DIR)}"
        f" --device {DEVICE}"
        f" --temperature {TEMPERATURE}"
        f" --top-p {TOP_P}"
        f" --top-k {int(TOP_K)}"
        f" --num-samples 1"
    )
    if MAX_NEW_TOKENS > 0:
        cmd_t2s += f" --max-new-tokens {int(MAX_NEW_TOKENS)}"
    if prompt_npy:
        cmd_t2s += f" --prompt-tokens {shlex.quote(prompt_npy)}"
        if PROMPT_TEXT.strip():
            cmd_t2s += f" --prompt-text {shlex.quote(PROMPT_TEXT)}"
    if COMPILE:
        cmd_t2s += " --compile"

    stage_label_2 = f"Chunk {chunk_num}/{num_chunks} — Text → Semantic Tokens  (4B LLM)"
    rc2, s2_time = run_stage(
        cmd_t2s, 2, stage_label_2,
        log_lines, t0,
        pct_start=0, pct_end=85,
        est_tokens=est_tokens,
        chunk_idx=chunk_num, total_chunks=num_chunks,
        overall_base=chunk_base_pct, overall_span=chunk_pct_each,
        chunk_eta_val=chunk_eta_val
    )

    # ── Stage 3: semantic tokens → WAV ───────────────────────
    codes = (sorted(glob.glob(os.path.join(OUTPUT_DIR, 'codes_*.npy')))
           + sorted(glob.glob(os.path.join(REPO_DIR,   'codes_*.npy'))))

    if not codes:
        upd(stage=2, stage_label=f"❌ codes_*.npy not found (Chunk {chunk_num})",
            log_lines=log_lines, elapsed=time.time() - t0,
            chunk_idx=chunk_num, total_chunks=num_chunks,
            stage_pct=100, overall_pct=chunk_base_pct + chunk_pct_each * 0.85,
            status='ERROR',
            error_msg=f"Stage 2 failed for chunk {chunk_num}. Check log.",
            cur_tps=0.0, cur_eta=0.0)
        raise FileNotFoundError(f"codes_*.npy not found for chunk {chunk_num}.")

    codes_file = codes[-1]
    sz_kb = os.path.getsize(codes_file) / 1024
    log_lines.append(
        f"[{time.strftime('%H:%M:%S')}] Using codes: {os.path.basename(codes_file)} ({sz_kb:.1f} KB)"
    )

    # Clean any previous fake.wav before decoding this chunk
    for _fw in [os.path.join(REPO_DIR, 'fake.wav'), os.path.join(OUTPUT_DIR, 'fake.wav')]:
        if os.path.exists(_fw):
            os.remove(_fw)

    # DAC decode: codes → WAV. Output goes to cwd as fake.wav
    cmd_dec = (
        f"{sys.executable} -m fish_speech.models.dac.inference"
        f" -i {shlex.quote(codes_file)}"
        f" --checkpoint-path {shlex.quote(os.path.join(MODEL_DIR, 'codec.pth'))}"
        f" --device {DEVICE}"
    )

    stage_label_3 = f"Chunk {chunk_num}/{num_chunks} — Semantic Tokens → WAV  (DAC decoder)"
    rc3, s3_time = run_stage(
        cmd_dec, 3, stage_label_3,
        log_lines, t0,
        pct_start=85, pct_end=100,
        est_tokens=400,
        chunk_idx=chunk_num, total_chunks=num_chunks,
        overall_base=chunk_base_pct, overall_span=chunk_pct_each,
        chunk_eta_val=chunk_eta_val
    )

    # Find the generated fake.wav — DAC writes it to cwd (REPO_DIR)
    import shutil as _shutil
    _fake_wav_repo = os.path.join(REPO_DIR, 'fake.wav')
    _fake_wav_out  = os.path.join(OUTPUT_DIR, 'fake.wav')
    chunk_wav_path = os.path.join(OUTPUT_DIR, f'chunk_{chunk_num:04d}.wav')

    if os.path.exists(_fake_wav_repo):
        _shutil.move(_fake_wav_repo, chunk_wav_path)
        chunk_wavs.append(chunk_wav_path)
        log_lines.append(f"[{time.strftime('%H:%M:%S')}] ✅ Chunk {chunk_num} saved: {os.path.basename(chunk_wav_path)}")
    elif os.path.exists(_fake_wav_out):
        _shutil.move(_fake_wav_out, chunk_wav_path)
        chunk_wavs.append(chunk_wav_path)
        log_lines.append(f"[{time.strftime('%H:%M:%S')}] ✅ Chunk {chunk_num} saved: {os.path.basename(chunk_wav_path)}")
    else:
        # Broader search as last resort
        wavs = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.wav')))
        wavs = [w for w in wavs if 'chunk_' not in os.path.basename(w)
                and 'output_s2pro' not in os.path.basename(w)]
        if wavs:
            _shutil.move(wavs[-1], chunk_wav_path)
            chunk_wavs.append(chunk_wav_path)
            log_lines.append(f"[{time.strftime('%H:%M:%S')}] ✅ Chunk {chunk_num} saved: {os.path.basename(chunk_wav_path)}")
        else:
            log_lines.append(f"[{time.strftime('%H:%M:%S')}] ⚠️  No WAV for chunk {chunk_num} — Stage 3 may have failed")

    chunk_elapsed = time.time() - chunk_t0
    chunk_times.append(chunk_elapsed)
    log_lines.append(f"[{time.strftime('%H:%M:%S')}] Chunk {chunk_num} done in {chunk_elapsed:.1f}s")

# ════════════════════════════════════════════════════════════
# CONCATENATE CHUNKS WITH CROSSFADE
# ════════════════════════════════════════════════════════════
total_elapsed = time.time() - t0

if not chunk_wavs:
    OUTPUT_WAV_PATH = ''
    upd(stage=3, stage_label="❌ No WAV chunks produced",
        log_lines=log_lines, elapsed=total_elapsed,
        stage_pct=99, overall_pct=99, status='ERROR',
        error_msg="Pipeline finished but no .wav chunks produced. Check log.",
        cur_tps=0.0, cur_eta=0.0)
    raise RuntimeError("No WAV output found. Check dashboard log above.")

log_lines.append(f"[{time.strftime('%H:%M:%S')}] Concatenating {len(chunk_wavs)} chunk(s)...")

import soundfile as sf

if len(chunk_wavs) == 1:
    # Single chunk — just rename
    import shutil as _shutil
    OUTPUT_WAV_PATH = os.path.join(OUTPUT_DIR, 'output_s2pro.wav')
    _shutil.move(chunk_wavs[0], OUTPUT_WAV_PATH)
else:
    # Crossfade concatenation
    all_audio = []
    target_sr = None
    for wpath in chunk_wavs:
        data, sr = sf.read(wpath)
        if data.ndim > 1:
            data = data[:, 0]  # mono
        if target_sr is None:
            target_sr = sr
        elif sr != target_sr:
            # Resample if needed
            import resampy
            data = resampy.resample(data, sr, target_sr)
        all_audio.append(data)

    # Crossfade
    crossfade_samples = int(target_sr * CROSSFADE_MS / 1000)
    if crossfade_samples < 1:
        crossfade_samples = 1

    result = all_audio[0]
    for i in range(1, len(all_audio)):
        nxt = all_audio[i]
        if len(result) >= crossfade_samples and len(nxt) >= crossfade_samples:
            fade_out = np.linspace(1, 0, crossfade_samples)
            fade_in  = np.linspace(0, 1, crossfade_samples)
            overlap = result[-crossfade_samples:] * fade_out + nxt[:crossfade_samples] * fade_in
            result = np.concatenate([result[:-crossfade_samples], overlap, nxt[crossfade_samples:]])
        else:
            result = np.concatenate([result, nxt])

    OUTPUT_WAV_PATH = os.path.join(OUTPUT_DIR, 'output_s2pro.wav')
    sf.write(OUTPUT_WAV_PATH, result, target_sr)

total_elapsed = time.time() - t0
wav_mb = os.path.getsize(OUTPUT_WAV_PATH) / 1e6
try:
    data_final, sr_final = sf.read(OUTPUT_WAV_PATH)
    duration = len(data_final) / sr_final
except Exception:
    duration, sr_final = 0, 44100

log_lines.append(
    f"[{time.strftime('%H:%M:%S')}] ✅ Final output: {os.path.basename(OUTPUT_WAV_PATH)}"
    f" ({wav_mb:.2f} MB, {duration:.1f}s)"
)
log_lines.append(f"[{time.strftime('%H:%M:%S')}] Total time: {total_elapsed:.0f}s")

upd(stage=3,
    stage_label=(
        f"✅ COMPLETE — {duration:.1f}s audio · {wav_mb:.2f} MB · {total_elapsed:.0f}s total"
    ),
    log_lines=log_lines, elapsed=total_elapsed,
    chunk_idx=num_chunks, total_chunks=num_chunks,
    tok_done=total_words * 25, tok_est=total_words * 25,
    stage_pct=100, overall_pct=100, status='DONE',
    cur_tps=0.0, cur_eta=0.0)

print(f"\n{SEP}")
print(f"  🎉 SYNTHESIS COMPLETE!")
print(f"{'─'*62}")
print(f"  Output       : {OUTPUT_WAV_PATH}")
print(f"  Size         : {wav_mb:.2f} MB")
print(f"  Duration     : {duration:.1f}s  ({duration/60:.1f} min)")
print(f"  Chunks       : {num_chunks}")
print(f"  Total time   : {total_elapsed:.0f}s  ({total_elapsed/60:.1f} min)")
if duration > 0:
    print(f"  RTF (GPU)    : {total_elapsed/duration:.2f}x realtime")
if chunk_times:
    avg_ct = sum(chunk_times) / len(chunk_times)
    print(f"  Avg chunk    : {avg_ct:.1f}s")
print(f"{SEP}")


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — PLAY & DOWNLOAD OUTPUT AUDIO
# ════════════════════════════════════════════════════════════
import os, glob
from IPython.display import Audio, display, HTML
from google.colab import files

SEP = "═" * 62
print(f"\n{SEP}")
print("  CELL 8 — PLAY & DOWNLOAD OUTPUT")
print(f"{SEP}\n")

# Find output wav
if 'OUTPUT_WAV_PATH' not in dir() or not OUTPUT_WAV_PATH:
    cands = (sorted(glob.glob('/content/inference_outputs/output_s2pro.wav'))
           + sorted(glob.glob('/content/inference_outputs/*.wav'))
           + sorted(glob.glob('/content/fish-speech/fake.wav')))
    if cands:
        OUTPUT_WAV_PATH = cands[0]
        print(f"   Found: {OUTPUT_WAV_PATH}")
    else:
        raise FileNotFoundError("No output WAV found. Run Cell 7 first.")

if not os.path.exists(OUTPUT_WAV_PATH):
    raise FileNotFoundError(f"WAV not found: {OUTPUT_WAV_PATH}")

wav_mb = os.path.getsize(OUTPUT_WAV_PATH) / 1e6

try:
    import soundfile as sf
    data, sr = sf.read(OUTPUT_WAV_PATH)
    duration = len(data) / sr
    channels = 'Stereo' if data.ndim > 1 else 'Mono'
except Exception as e:
    data, sr, duration, channels = None, 44100, 0, 'Unknown'
    print(f"   ⚠️  Could not read audio metadata: {e}")

print(f"  📄 File     : {OUTPUT_WAV_PATH}")
print(f"  📦 Size     : {wav_mb:.2f} MB")
print(f"  ⏱  Duration : {duration:.2f}s  ({duration/60:.2f} min)")
print(f"  🎵 Rate     : {sr} Hz  |  {channels}\n")

# ── Waveform + spectrogram ───────────────────────────────────
try:
    import matplotlib, numpy as np
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    audio_mono = data[:,0] if (data is not None and data.ndim > 1) else data
    if audio_mono is not None and len(audio_mono) > 0:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 5))
        fig.patch.set_facecolor('#080c18')
        for ax in (ax1, ax2):
            ax.set_facecolor('#0c1525')
            for sp in ax.spines.values():
                sp.set_color('#162840')
            ax.tick_params(colors='#3a6080', labelsize=8)

        t = np.linspace(0, duration, len(audio_mono))
        ax1.plot(t, audio_mono, lw=0.35, color='#00d4ff', alpha=0.9)
        ax1.set_ylabel('Amplitude', color='#3a6080', fontsize=9)
        ax1.set_title(f'Waveform  ·  {sr} Hz  ·  {duration:.1f}s  ·  {channels}',
                       color='#00d4ff', fontsize=10, pad=6)
        ax1.set_xlim(0, duration)
        ax1.grid(True, alpha=0.12, color='#162840')

        spec = audio_mono[:min(len(audio_mono), sr*45)]
        ax2.specgram(spec, Fs=sr, cmap='plasma', NFFT=1024, noverlap=512)
        ax2.set_ylabel('Freq (Hz)', color='#3a6080', fontsize=9)
        ax2.set_xlabel('Time (s)', color='#3a6080', fontsize=9)
        ax2.set_title('Spectrogram', color='#ff9f43', fontsize=10, pad=6)
        ax2.set_ylim(0, min(8000, sr // 2))

        fig.suptitle('🐟 fishaudio/s2-pro  ─  Generated Audio',
                     color='#7fff00', fontsize=12, fontweight='bold', y=1.01)
        plt.tight_layout()
        outpng = os.path.join(OUTPUT_DIR, 'waveform.png')
        plt.savefig(outpng, dpi=130, bbox_inches='tight', facecolor='#080c18')
        plt.show()
        print("   ✅ Waveform + spectrogram plotted above.")
    else:
        print("   ⚠️  No audio data to plot.")
except Exception as e:
    print(f"   ⚠️  Plot failed: {e}")

# ── Audio player ─────────────────────────────────────────────
print("\n🔊 Audio player (press ▶ to listen):")
display(HTML("""
<div style="background:#080c18;border:1px solid #00d4ff;border-radius:6px;
   padding:10px 14px;font-family:monospace;color:#00d4ff;margin:6px 0;
   display:inline-block">
  ▶ Press play to listen to your S2-Pro generated audio
</div>"""))
display(Audio(OUTPUT_WAV_PATH, autoplay=False))

# ── Download ─────────────────────────────────────────────────
print("\n💾 Downloading to your local machine...")
files.download(OUTPUT_WAV_PATH)

print(f"\n{SEP}")
print("  🎉 ALL DONE!")
print(f"{'─'*62}")
print(f"  ✅ Audio is downloading to your browser.")
print(f"\n  If download didn't start automatically:")
print(f"  → Sidebar 📁 → inference_outputs/ → right-click → Download")
print(f"  → Or run:  files.download('{OUTPUT_WAV_PATH}')")
print(f"{SEP}")
